In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
from deal_with_data import load_images_from_folder, extrapolate_SE3
from read_colmap_data import ColmapModel
from transformtion import flatten_params, unflatten_params
from RSSC_TE import optimize_RS_masked_manifold_TE
from RSSC_CEQ import optimize_RS_masked_manifold_CEQ
from RSSC_CEH import optimize_RS_masked_manifold_CEH
from RSSC_DPQ import optimize_RS_masked_manifold_DPQ
from RSSC_DPH import optimize_RS_masked_manifold_DPH
from tool import filter_points_by_distance, find_corresponding_points, find_corresponding_points_seq

In [2]:
def main_RSSC_TE():

    image_path = './datasets/data1/images'
    colmap_result_path = './datasets/data1/colmap_result'

    gamma = 0.5
    optimize_sample = 500  # number of sampled points
    np.random.seed(388)     
    max_nfev=50

    # ==================== Load the COLMAP initialization results ====================
    print("\n[1/5] Loading COLMAP results......")
    all_images_rs = load_images_from_folder(image_path)
    colmap_result = ColmapModel(str(colmap_result_path), str(image_path))
    # Read the initial camera intrinsics
    K = colmap_result.read_camera()
    print(f"COLMAP initial intrinsics K:\n{K}")
    # Read the initial camera poses
    all_R, all_T = colmap_result.read_images()
    all_pose = []
    for R_mat, t in zip(all_R, all_T):
        T = np.eye(4)
        T[:3,:3] = R_mat
        T[:3,3] = t
        all_pose.append(T)
    all_pose = np.array(all_pose)
    # Read the initial 3D points and 2D points
    points3D_xyz, points2D_uv, mask = colmap_result.read_points_and_2d()
    # Read all images
    all_image = colmap_result.read_all_images()
    H, W, _ = all_image[0].shape

    # ==================== Data sampling ====================
    print(f"\n[2/5] Data sampling (sampling {optimize_sample} points)...")
    N_points = points3D_xyz.shape[0]
    sampled_idx = np.random.choice(N_points, size=optimize_sample, replace=False)
    points3D_xyz = points3D_xyz[sampled_idx]                
    points2D_uv = points2D_uv[:, sampled_idx, :]           
    mask = mask[:, sampled_idx]    

    # ==================== Build the optimization variables ====================
    print(f"\n[3/5] Preparing data as optimization variables...")       
    all_pose_TE = extrapolate_SE3(all_pose, a=1, b=2)
    all_x, meta = flatten_params(all_pose_TE, points3D_xyz, K, gamma)
    x_if = np.ones_like(all_x, dtype=bool)
    x_if[6:12] = False  # fix the parameters of one camera
    mask_rs_TE = mask

    # ==================== Start optimization ====================
    print(f"\n[4/5] Optimization started...") 
    result_TE, x_opt_TE = optimize_RS_masked_manifold_TE(all_x, x_if, meta, H, W, points2D_uv, mask_rs_TE, max_nfev) 

    # ==================== Output results ====================
    print(f"\n[5/5] Outputting results...") 
    fx, fy, cx, cy, gamma_opt = x_opt_TE[-5:]
    K_opt = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    print(f"Optimized intrinsics K:\n{K_opt}")    
    print(f"Optimized gamma: {gamma_opt}")


In [3]:
main_RSSC_TE()


[1/5] Loading COLMAP results......
Loaded 50 images in total
COLMAP initial intrinsics K:
[[707.03978581   0.         640.        ]
 [  0.         707.03978581 512.        ]
 [  0.           0.           1.        ]]

[2/5] Data sampling (sampling 500 points)...

[3/5] Preparing data as optimization variables...

[4/5] Optimization started...
0 : [[5115965.1036314]]
1 : total_cost=134864.0494, fx=738.8860, fy=733.4688, cx=630.9808, cy=526.4305, gamma=0.4239
2 : total_cost=5181.0561, fx=742.2829, fy=742.4890, cx=627.1858, cy=517.4097, gamma=0.5274
3 : total_cost=2187.5888, fx=742.5958, fy=742.1121, cx=628.2578, cy=515.4774, gamma=0.5307
4 : total_cost=2176.6613, fx=742.8942, fy=742.5864, cx=628.2126, cy=515.3344, gamma=0.5331
5 : total_cost=2176.5853, fx=743.0051, fy=742.6920, cx=628.2279, cy=515.2452, gamma=0.5336
6 : total_cost=2176.5812, fx=743.0160, fy=742.7118, cx=628.2264, cy=515.2398, gamma=0.5337
7 : total_cost=2176.5810, fx=743.0224, fy=742.7187, cx=628.2269, cy=515.2353, gamm

In [4]:
def main_RSSC_CEQ():

    image_path = './datasets/data1/images'
    colmap_result_path = './datasets/data1/colmap_result'

    gamma = 0.5
    optimize_sample = 500  # number of sampled points
    np.random.seed(388)     
    max_nfev=50

    # ==================== Load the COLMAP initialization results ====================
    print("\n[1/6] Loading COLMAP results......")
    all_images_rs = load_images_from_folder(image_path)
    colmap_result = ColmapModel(str(colmap_result_path), str(image_path))
    # Read the initial camera intrinsics
    K = colmap_result.read_camera()
    print(f"COLMAP initial intrinsics K:\n{K}")
    # Read the initial camera poses
    all_R, all_T = colmap_result.read_images()
    all_pose = []
    for R_mat, t in zip(all_R, all_T):
        T = np.eye(4)
        T[:3,:3] = R_mat
        T[:3,3] = t
        all_pose.append(T)
    all_pose = np.array(all_pose)
    # Read the initial 3D points and 2D points
    points3D_xyz, points2D_uv, mask = colmap_result.read_points_and_2d()
    # Read all images
    all_image = colmap_result.read_all_images()
    H, W, _ = all_image[0].shape

    # ==================== Data sampling ====================
    print(f"\n[2/6] Data sampling (sampling {optimize_sample} points)...")
    N_points = points3D_xyz.shape[0]
    sampled_idx = np.random.choice(N_points, size=optimize_sample, replace=False)
    points3D_xyz = points3D_xyz[sampled_idx]                
    points2D_uv = points2D_uv[:, sampled_idx, :]           
    mask = mask[:, sampled_idx]  


    # ==================== Find point correspondences ====================
    print(f"\n[3/6] Finding point correspondences between neighboring frames...")
    tracked_back, tracked_forward, mask_rs_CEQ = find_corresponding_points_seq(
        all_images_rs, points2D_uv, mask, k_back=(1,), k_forward=(1,), threshold=0.1)
    points2D_back1, _ = tracked_back[1]
    points2D_forward1, _ = tracked_forward[1]

    # ==================== Build the optimization variables ====================
    print(f"\n[4/6] Preparing data as optimization variables...")   
    all_pose_CEQ = all_pose[2:-1, :, :]
    all_x, meta = flatten_params(all_pose_CEQ, points3D_xyz, K, gamma)
    x_if = np.ones_like(all_x, dtype=bool)
    x_if[6:12] = False  
    t_target = 1

    # ==================== Start optimization ====================
    print(f"\n[5/6] Optimization started...") 
    result_CEQ, x_opt_CEQ = optimize_RS_masked_manifold_CEQ(all_x, x_if, meta, H, W, points2D_uv, points2D_back1, points2D_forward1, mask_rs_CEQ, t_target, max_nfev)

    # ==================== Output results ====================
    print(f"\n[6/6] Outputting results...") 
    fx, fy, cx, cy, gamma_opt = x_opt_CEQ[-5:]
    K_opt = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    print(f"Optimized intrinsics K:\n{K_opt}")    
    print(f"Optimized gamma: {gamma_opt}")

In [5]:
main_RSSC_CEQ()


[1/6] Loading COLMAP results......
Loaded 50 images in total
COLMAP initial intrinsics K:
[[707.03978581   0.         640.        ]
 [  0.         707.03978581 512.        ]
 [  0.           0.           1.        ]]

[2/6] Data sampling (sampling 500 points)...

[3/6] Finding point correspondences between neighboring frames...

[4/6] Preparing data as optimization variables...

[5/6] Optimization started...
0 : [[4627742.5006053]]
1 : total_cost=113839.3974, fx=737.9594, fy=731.1761, cx=630.6028, cy=525.5152, gamma=0.4327
2 : total_cost=2774.1690, fx=743.8624, fy=744.4568, cx=627.1447, cy=517.3166, gamma=0.5439
3 : total_cost=1760.9073, fx=744.2925, fy=744.1681, cx=628.8114, cy=515.1858, gamma=0.5481
4 : total_cost=1760.0251, fx=744.4385, fy=744.4308, cx=628.7909, cy=515.1501, gamma=0.5492
5 : total_cost=1760.0227, fx=744.4463, fy=744.4191, cx=628.7964, cy=515.1341, gamma=0.5492
6 : total_cost=1760.0226, fx=744.4487, fy=744.4249, cx=628.7962, cy=515.1345, gamma=0.5492
Finished (tolFu

In [6]:
def main_RSSC_CEH():

    image_path = './datasets/data1/images'
    colmap_result_path = './datasets/data1/colmap_result'

    gamma = 0.5
    optimize_sample = 500  # number of sampled points
    np.random.seed(388)     
    max_nfev=50

    # ==================== Load the COLMAP initialization results ====================
    print("\n[1/6] Loading COLMAP results......")
    all_images_rs = load_images_from_folder(image_path)
    colmap_result = ColmapModel(str(colmap_result_path), str(image_path))
    # Read the initial camera intrinsics
    K = colmap_result.read_camera()
    print(f"COLMAP initial intrinsics K:\n{K}")
    # Read the initial camera poses
    all_R, all_T = colmap_result.read_images()
    all_pose = []
    for R_mat, t in zip(all_R, all_T):
        T = np.eye(4)
        T[:3,:3] = R_mat
        T[:3,3] = t
        all_pose.append(T)
    all_pose = np.array(all_pose)
    # Read the initial 3D points and 2D points
    points3D_xyz, points2D_uv, mask = colmap_result.read_points_and_2d()
    # Read all images
    all_image = colmap_result.read_all_images()
    H, W, _ = all_image[0].shape

    # ==================== Data sampling ====================
    print(f"\n[2/6] Data sampling (sampling {optimize_sample} points)...")
    N_points = points3D_xyz.shape[0]
    sampled_idx = np.random.choice(N_points, size=optimize_sample, replace=False)
    points3D_xyz = points3D_xyz[sampled_idx]                
    points2D_uv = points2D_uv[:, sampled_idx, :]           
    mask = mask[:, sampled_idx]  


    # ==================== Find point correspondences ====================
    print(f"\n[3/6] Finding point correspondences between neighboring frames...")
    tracked_back, tracked_forward, mask_rs_CEH = find_corresponding_points_seq(
        all_images_rs, points2D_uv, mask, k_back=(1, 2), k_forward=(1,), threshold=0.1)
    points2D_back1, _ = tracked_back[1]
    points2D_back2, _ = tracked_back[2]
    points2D_forward1, _ = tracked_forward[1]

    # ==================== Build the optimization variables ====================
    print(f"\n[4/6] Preparing data as optimization variables...")   
    all_pose_CEH = all_pose[2:-1, :, :]
    all_x, meta = flatten_params(all_pose_CEH, points3D_xyz, K, gamma)
    x_if = np.ones_like(all_x, dtype=bool)
    x_if[6:12] = False  
    t_target = 2

    # ==================== Start optimization ====================
    print(f"\n[5/6] Optimization started...") 
    result_CEH, x_opt_CEH = optimize_RS_masked_manifold_CEH(all_x, x_if, meta, H, W, points2D_uv,points2D_back2, points2D_back1,points2D_forward1, mask_rs_CEH, t_target, max_nfev)

    # ==================== Output results ====================
    print(f"\n[6/6] Outputting results...") 
    fx, fy, cx, cy, gamma_opt = x_opt_CEH[-5:]
    K_opt = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    print(f"Optimized intrinsics K:\n{K_opt}")    
    print(f"Optimized gamma: {gamma_opt}")

In [7]:
main_RSSC_CEH()


[1/6] Loading COLMAP results......
Loaded 50 images in total
COLMAP initial intrinsics K:
[[707.03978581   0.         640.        ]
 [  0.         707.03978581 512.        ]
 [  0.           0.           1.        ]]

[2/6] Data sampling (sampling 500 points)...

[3/6] Finding point correspondences between neighboring frames...

[4/6] Preparing data as optimization variables...

[5/6] Optimization started...
0 : [[3086515.98395754]]
1 : total_cost=83582.2736, fx=737.4847, fy=732.2129, cx=628.9877, cy=524.2104, gamma=0.4423
2 : total_cost=1801.3637, fx=744.3268, fy=745.0215, cx=627.3174, cy=517.2214, gamma=0.5343
3 : total_cost=1421.5685, fx=744.8813, fy=745.2432, cx=628.2417, cy=515.9774, gamma=0.5405
4 : total_cost=1421.2324, fx=745.0461, fy=745.4894, cx=628.2144, cy=515.9343, gamma=0.5416
5 : total_cost=1421.2305, fx=745.0535, fy=745.4747, cx=628.2203, cy=515.9124, gamma=0.5415
6 : total_cost=1421.2304, fx=745.0568, fy=745.4804, cx=628.2195, cy=515.9120, gamma=0.5416
Finished (tolFu

In [8]:
def main_RSSC_DPQ():

    image_path = './datasets/data1/images'
    colmap_result_path = './datasets/data1/colmap_result'

    gamma = 0.5
    optimize_sample = 500  # number of sampled points
    np.random.seed(388)     
    max_nfev=50

    # ==================== Load the COLMAP initialization results ====================
    print("\n[1/6] Loading COLMAP results......")
    all_images_rs = load_images_from_folder(image_path)
    colmap_result = ColmapModel(str(colmap_result_path), str(image_path))
    # Read the initial camera intrinsics
    K = colmap_result.read_camera()
    print(f"COLMAP initial intrinsics K:\n{K}")
    # Read the initial camera poses
    all_R, all_T = colmap_result.read_images()
    all_pose = []
    for R_mat, t in zip(all_R, all_T):
        T = np.eye(4)
        T[:3,:3] = R_mat
        T[:3,3] = t
        all_pose.append(T)
    all_pose = np.array(all_pose)
    # Read the initial 3D points and 2D points
    points3D_xyz, points2D_uv, mask = colmap_result.read_points_and_2d()
    # Read all images
    all_image = colmap_result.read_all_images()
    H, W, _ = all_image[0].shape

    # ==================== Data sampling ====================
    print(f"\n[2/6] Data sampling (sampling {optimize_sample} points)...")
    N_points = points3D_xyz.shape[0]
    sampled_idx = np.random.choice(N_points, size=optimize_sample, replace=False)
    points3D_xyz = points3D_xyz[sampled_idx]                
    points2D_uv = points2D_uv[:, sampled_idx, :]           
    mask = mask[:, sampled_idx]  

    # ==================== Find point correspondences ====================
    print(f"\n[3/6] Finding point correspondences between neighboring frames...")
    tracked_back, tracked_forward, mask_rs_DPQ = find_corresponding_points_seq(
        all_images_rs, points2D_uv, mask, k_back=(1,), k_forward=(1,), threshold=0.1)
    points2D_back1, _ = tracked_back[1]
    points2D_forward1, _ = tracked_forward[1]

    # ==================== Build the optimization variables ====================
    print(f"\n[4/6] Preparing data as optimization variables...")   
    all_pose_DPQ = extrapolate_SE3(all_pose, a=1, b=2)
    all_x, meta = flatten_params(all_pose_DPQ, points3D_xyz, K, gamma)
    x_if = np.ones_like(all_x, dtype=bool)
    x_if[6:12] = False  
    t_target = 1.5
    mask_rs_TE = mask

    # ==================== Start optimization ====================
    print(f"\n[5/6] Optimization started...") 
    result_DPQ, x_opt_DPQ = optimize_RS_masked_manifold_DPQ(all_x, x_if, meta, H, W, points2D_uv, points2D_back1,points2D_forward1, mask_rs_TE, mask_rs_DPQ, t_target, max_nfev)

    # ==================== Output results ====================
    print(f"\n[6/6] Outputting results...") 
    fx, fy, cx, cy, gamma_opt = x_opt_DPQ[-5:]
    K_opt = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    print(f"Optimized intrinsics K:\n{K_opt}")    
    print(f"Optimized gamma: {gamma_opt}")

In [9]:
main_RSSC_DPQ()


[1/6] Loading COLMAP results......
Loaded 50 images in total
COLMAP initial intrinsics K:
[[707.03978581   0.         640.        ]
 [  0.         707.03978581 512.        ]
 [  0.           0.           1.        ]]

[2/6] Data sampling (sampling 500 points)...

[3/6] Finding point correspondences between neighboring frames...

[4/6] Preparing data as optimization variables...

[5/6] Optimization started...
0 : [[6268791.85442436]]
1 : total_cost=169621.7617, fx=741.3309, fy=733.4574, cx=631.2770, cy=525.5324, gamma=0.4476
2 : total_cost=6218.4649, fx=742.5509, fy=742.9793, cx=627.3410, cy=517.0816, gamma=0.5322
3 : total_cost=2873.4078, fx=743.1853, fy=743.1116, cx=628.2776, cy=515.4980, gamma=0.5407
4 : total_cost=2866.6067, fx=743.4772, fy=743.5725, cx=628.2387, cy=515.3435, gamma=0.5430
5 : total_cost=2866.5267, fx=743.5809, fy=743.6899, cx=628.2485, cy=515.2711, gamma=0.5435
6 : total_cost=2866.5223, fx=743.5924, fy=743.7106, cx=628.2474, cy=515.2658, gamma=0.5436
7 : total_cost

In [10]:
def main_RSSC_DPH():

    image_path = './datasets/data1/images'
    colmap_result_path = './datasets/data1/colmap_result'

    gamma = 0.5
    optimize_sample = 500  # number of sampled points
    np.random.seed(388)     
    max_nfev=50

    # ==================== Load the COLMAP initialization results ====================
    print("\n[1/6] Loading COLMAP results......")
    all_images_rs = load_images_from_folder(image_path)
    colmap_result = ColmapModel(str(colmap_result_path), str(image_path))
    # Read the initial camera intrinsics
    K = colmap_result.read_camera()
    print(f"COLMAP initial intrinsics K:\n{K}")
    # Read the initial camera poses
    all_R, all_T = colmap_result.read_images()
    all_pose = []
    for R_mat, t in zip(all_R, all_T):
        T = np.eye(4)
        T[:3,:3] = R_mat
        T[:3,3] = t
        all_pose.append(T)
    all_pose = np.array(all_pose)
    # Read the initial 3D points and 2D points
    points3D_xyz, points2D_uv, mask = colmap_result.read_points_and_2d()
    # Read all images
    all_image = colmap_result.read_all_images()
    H, W, _ = all_image[0].shape

    # ==================== Data sampling ====================
    print(f"\n[2/6] Data sampling (sampling {optimize_sample} points)...")
    N_points = points3D_xyz.shape[0]
    sampled_idx = np.random.choice(N_points, size=optimize_sample, replace=False)
    points3D_xyz = points3D_xyz[sampled_idx]                
    points2D_uv = points2D_uv[:, sampled_idx, :]           
    mask = mask[:, sampled_idx]  


    # ==================== Find point correspondences ====================
    print(f"\n[3/6] Finding point correspondences between neighboring frames...")
    tracked_back, tracked_forward, mask_rs_DPH = find_corresponding_points_seq(
        all_images_rs, points2D_uv, mask, k_back=(1, 2), k_forward=(1,), threshold=0.1)
    points2D_back1, _ = tracked_back[1]
    points2D_back2, _ = tracked_back[2]
    points2D_forward1, _ = tracked_forward[1]

    # ==================== Build the optimization variables ====================
    print(f"\n[4/6] Preparing data as optimization variables...")   
    all_pose_DPH = extrapolate_SE3(all_pose, a=1, b=2)
    all_x, meta = flatten_params(all_pose_DPH, points3D_xyz, K, gamma)
    x_if = np.ones_like(all_x, dtype=bool)
    x_if[6:12] = False  
    t_target = 2
    mask_rs_TE = mask

    # ==================== Start optimization ====================
    print(f"\n[5/6] Optimization started...") 
    result_DPH, x_opt_DPH = optimize_RS_masked_manifold_DPH(all_x, x_if, meta, H, W, points2D_uv, points2D_back2, points2D_back1,points2D_forward1, mask_rs_TE, mask_rs_DPH, t_target, max_nfev)

    # ==================== Output results ====================
    print(f"\n[6/6] Outputting results...") 
    fx, fy, cx, cy, gamma_opt = x_opt_DPH[-5:]
    K_opt = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    print(f"Optimized intrinsics K:\n{K_opt}")    
    print(f"Optimized gamma: {gamma_opt}")

In [11]:
main_RSSC_DPH()


[1/6] Loading COLMAP results......
Loaded 50 images in total
COLMAP initial intrinsics K:
[[707.03978581   0.         640.        ]
 [  0.         707.03978581 512.        ]
 [  0.           0.           1.        ]]

[2/6] Data sampling (sampling 500 points)...

[3/6] Finding point correspondences between neighboring frames...

[4/6] Preparing data as optimization variables...

[5/6] Optimization started...
0 : [[5858153.0792539]]
1 : total_cost=159163.5853, fx=740.7730, fy=733.4752, cx=631.0595, cy=525.6382, gamma=0.4430
2 : total_cost=5370.9902, fx=741.4794, fy=742.0283, cx=627.3842, cy=517.4819, gamma=0.5184
3 : total_cost=2739.1859, fx=742.0355, fy=741.3379, cx=628.2608, cy=515.5863, gamma=0.5239
4 : total_cost=2733.6287, fx=742.3115, fy=741.7593, cx=628.2261, cy=515.4606, gamma=0.5264
5 : total_cost=2733.5305, fx=742.4242, fy=741.8508, cx=628.2435, cy=515.3625, gamma=0.5269
6 : total_cost=2733.5241, fx=742.4379, fy=741.8720, cx=628.2425, cy=515.3547, gamma=0.5270
7 : total_cost=